# Time-varying CN↔US Citation Trend via Yearly Logistic Fits

This notebook studies how the citation mechanism between Chinese and American papers varies over time. For each `source_year = y`, we fit logistic regression models under **3-year**, **5-year**, and **10-year** target windows to extract pair-specific similarity slopes `β(sim × CN→US)` and `β(sim × US→CN)`, and track their evolution.

## §0 Project file inspection & implementation plan

### Files inspected
| File | Status | Purpose |
|------|--------|---------|
| `data/dataCleanSCIE.csv` | exists | 25,794 papers; key cols: UT, PY, country, CR, DI |
| `output/cluster_results/specter2_embeddings_cache.npy` | exists | (25794, 768) SPECTER2 embeddings, row-aligned with CSV |
| `output/citation_graph/edges.parquet` | exists | Citation edges: src, dst, country_src, country_dst (normalized CN/US/Other) |
| `output/cluster_results/paper_topics.csv` | exists | paper_id to topic mapping |

### Implementation plan
1. **Load raw CSV** and normalize country to `{CN, US, Other}` using `CN_LABELS_SET` / `US_LABELS_SET`.
2. **Load embeddings** and L2-normalize; align with CSV rows by index.
3. **Load citation edges** from `output/citation_graph/edges.parquet` and filter to CN/US endpoints.
4. **Load topic mapping** and merge into df.
5. For each `source_year in [2005, 2024]` x `window in {3, 5, 10}`:
   - Define S_y (source papers of year y) and D_{y,w} (target pool [y-w+1, y])
   - **Restricted FAISS similarity search** (top-30) within target pool
   - Build union edge set (sim candidates + citation edges), compute features
   - **Matched negative sampling** (1:1, stratified by country_pair x year_gap_bin)
   - Fit `LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")` on **raw cosine sim**
   - Extract beta(sim x CN->US), beta(sim x US->CN), gap
6. Export CSVs and time-series plots.

### Model form
logit(P(cite)) = a0 + a_CNUS * I(CN->US) + a_USCN * I(US->CN) + a_USUS * I(US->US)
  + g1 * year_gap_norm + g2 * same_topic
  + beta_CNCN * (sim * I(CN->CN)) + beta_CNUS * (sim * I(CN->US))
  + beta_USCN * (sim * I(US->CN)) + beta_USUS * (sim * I(US->US))

**No z-score normalization** of sim -- raw cosine enters the model for cross-year comparability.
**No bootstrap / no uncertainty** -- only deterministic point estimates.
**Feature standardization** is applied internally before fitting to prevent numerical overflow (quasi-complete separation); coefficients are then converted back to the original scale.

## §1 Load reusable inputs

In [43]:
import warnings, pathlib, time
import numpy as np
import pandas as pd
import faiss
from sklearn.linear_model import LogisticRegression
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ─────────────────────────────────────────────────────────────────
ROOT        = pathlib.Path(".")
DATA_CSV    = ROOT / "data" / "dataCleanSCIE.csv"
EMB_NPY     = ROOT / "output" / "cluster_results" / "specter2_embeddings_cache.npy"
CITE_EDGES  = ROOT / "output" / "citation_graph" / "edges.parquet"
TOPICS_CSV  = ROOT / "output" / "cluster_results" / "paper_topics.csv"
OUT_TABLE   = ROOT / "output" / "compare_networks" / "tables"
OUT_FIG     = ROOT / "output" / "compare_networks" / "figs"
OUT_TABLE.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

# ── Configurable constants ────────────────────────────────────────────────
TOPK_RESTRICTED  = 30
NEG_RATIO        = 1        # negative : positive
MIN_POS_TOTAL    = 30
MIN_POS_PAIR     = 5
SOURCE_YEAR_MIN  = 2005
SOURCE_YEAR_MAX  = 2024
ALLOW_SAME_YEAR  = True
SEED             = 42

# ── Country normalisation ─────────────────────────────────────────────────
CN_LABELS_SET = {
    "China", "china", "CN", "CHINA",
    "Peoples R China", "Peoples R. China", "PEOPLES R CHINA",
    "PRC", "prc", "PR China", "PR CHINA",
}
US_LABELS_SET = {
    "USA", "US", "U.S.A.", "U.S.",
    "United States", "united states",
    "United States of America",
    "UNITED STATES", "UNITED STATES OF AMERICA",
}
_CN_UPPER = {x.upper() for x in CN_LABELS_SET}
_US_UPPER = {x.upper() for x in US_LABELS_SET}

def map_country(c) -> str:
    if pd.isna(c) or not str(c).strip():
        return "Other"
    cu = str(c).strip().upper()
    if cu in _CN_UPPER:
        return "CN"
    if cu in _US_UPPER:
        return "US"
    return "Other"

print("Config ready.")

Config ready.


In [44]:
# ── 1a  Load raw papers ────────────────────────────────────────────────────
t0 = time.time()
df = pd.read_csv(DATA_CSV, low_memory=False)
print(f"Raw CSV: {len(df):,} rows x {df.shape[1]} cols  ({time.time()-t0:.1f}s)")

df["paper_id"]  = df["UT"].astype(str).str.strip()
df["year"]      = pd.to_numeric(df["PY"], errors="coerce").astype("Int64")
df["country2"]  = df["country"].apply(map_country)

print(f"Year range : {df['year'].min()} – {df['year'].max()}")
print(f"Country2   : {df['country2'].value_counts().to_dict()}")

Raw CSV: 25,794 rows x 41 cols  (0.7s)
Year range : 1990 – 2026
Country2   : {'Other': 16728, 'CN': 5687, 'US': 3379}


In [45]:
# ── 1b  Load embeddings & L2-normalise ────────────────────────────────────
emb = np.load(EMB_NPY).astype(np.float32)
n_align = min(len(df), emb.shape[0])
df  = df.iloc[:n_align].reset_index(drop=True)
emb = emb[:n_align]

norms = np.linalg.norm(emb, axis=1, keepdims=True)
norms[norms == 0] = 1e-10
emb = emb / norms

print(f"Embeddings : {emb.shape}  (aligned with df: {len(df):,} rows)")
assert len(df) == emb.shape[0], "Row mismatch between df and embeddings!"

Embeddings : (25794, 768)  (aligned with df: 25,794 rows)


In [46]:
# ── 1c  Load citation edges (CN/US only) ──────────────────────────────────
cite_all = pd.read_parquet(CITE_EDGES)
print(f"Citation edges total: {len(cite_all):,}")
print(f"Columns: {list(cite_all.columns)}")

# Keep only edges where both endpoints are CN or US
cite_cnus = cite_all[
    cite_all["country_src"].isin({"CN", "US"}) &
    cite_all["country_dst"].isin({"CN", "US"})
].copy()
print(f"CN/US internal citation edges: {len(cite_cnus):,}")

# Build a fast lookup set:  (src_paper_id, dst_paper_id)
cite_set = set(zip(cite_cnus["src"], cite_cnus["dst"]))

Citation edges total: 49,396
Columns: ['src', 'dst', 'match_type', 'year_src', 'year_dst', 'country_src', 'country_dst']
CN/US internal citation edges: 11,199


In [47]:
# ── 1d  Load topic mapping ─────────────────────────────────────────────────
topics = pd.read_csv(TOPICS_CSV)
topics = topics[["paper_id", "topic"]].drop_duplicates(subset="paper_id")
df = df.merge(topics, on="paper_id", how="left")
df["topic"] = df["topic"].fillna(-1).astype(int)

print(f"Topic coverage: {(df['topic'] != -1).sum():,} / {len(df):,}")

# ── Build lookup dicts ────────────────────────────────────────────────────
pid_to_idx     = dict(zip(df["paper_id"], df.index))
pid_to_year    = dict(zip(df["paper_id"], df["year"]))
pid_to_country = dict(zip(df["paper_id"], df["country2"]))
pid_to_topic   = dict(zip(df["paper_id"], df["topic"]))
idx_to_pid     = dict(zip(df.index, df["paper_id"]))

# Precompute CN/US paper indices per year for fast access
from collections import defaultdict
year_country_indices = defaultdict(list)  # (year, country) -> [idx, ...]
for idx, row in df.iterrows():
    if row["country2"] in ("CN", "US") and pd.notna(row["year"]):
        year_country_indices[(int(row["year"]), row["country2"])].append(idx)

# Also a combined index: year -> all CN/US indices
year_cnus_indices = defaultdict(list)
for (y, c), idxs in year_country_indices.items():
    year_cnus_indices[y].extend(idxs)

data_year_min = int(df.loc[df["country2"].isin({"CN", "US"}), "year"].min())
data_year_max = int(df.loc[df["country2"].isin({"CN", "US"}), "year"].max())
print(f"CN/US year range: {data_year_min} – {data_year_max}")
print(f"§1 complete.")

Topic coverage: 14,299 / 25,794
CN/US year range: 1990 – 2026
§1 complete.


## §2 Helper functions

Core routines for:
1. Selecting source & target paper sets per (year, window)
2. Restricted FAISS similarity search within target pool
3. Building the union edge set with features
4. Matched negative sampling
5. Fitting the logistic model and extracting betas

In [48]:
def get_source_and_target(source_year, window):
    """Return (source_indices, target_indices) as sorted numpy arrays."""
    y = source_year
    w = window
    y_start = max(y - w + 1, data_year_min)
    y_end   = y  # inclusive

    # Source: papers of year y, country in {CN, US}
    src_idxs = np.array(year_cnus_indices.get(y, []), dtype=np.int64)

    # Target: papers in [y_start, y_end], country in {CN, US}
    tgt_idxs = []
    for yy in range(y_start, y_end + 1):
        tgt_idxs.extend(year_cnus_indices.get(yy, []))
    tgt_idxs = np.array(sorted(set(tgt_idxs)), dtype=np.int64)

    return src_idxs, tgt_idxs


def restricted_sim_search(src_idxs, tgt_idxs, topk=TOPK_RESTRICTED):
    """FAISS inner-product search: query=src embeddings, index=tgt embeddings.
    Returns DataFrame with columns [src_idx, dst_idx, sim] (global indices).
    Self-loops removed.
    """
    if len(src_idxs) == 0 or len(tgt_idxs) == 0:
        return pd.DataFrame(columns=["src_idx", "dst_idx", "sim"])

    emb_tgt = emb[tgt_idxs]
    emb_src = emb[src_idxs]

    k_actual = min(topk + 1, len(tgt_idxs))  # +1 for potential self
    index = faiss.IndexFlatIP(emb_tgt.shape[1])
    index.add(emb_tgt)
    sims, ids = index.search(emb_src, k_actual)

    rows = []
    for qi in range(len(src_idxs)):
        s_global = src_idxs[qi]
        for ki in range(k_actual):
            if ids[qi, ki] < 0:
                continue
            d_global = tgt_idxs[ids[qi, ki]]
            if d_global == s_global:  # self-loop
                continue
            rows.append((int(s_global), int(d_global), float(sims[qi, ki])))
            if len(rows) > 0 and rows[-1] != rows[-1]:  # guard
                continue

    out = pd.DataFrame(rows, columns=["src_idx", "dst_idx", "sim"])
    # Deduplicate (keep highest sim per pair)
    out = out.sort_values("sim", ascending=False).drop_duplicates(
        subset=["src_idx", "dst_idx"], keep="first"
    )
    return out


def build_union_edges(sim_edges, src_idxs, tgt_idxs, window):
    """Build the union edge set from similarity candidates + citation edges.
    Returns DataFrame with: src_idx, dst_idx, src_pid, dst_pid, sim, label,
    year_gap, year_gap_norm, same_topic, country_pair.
    """
    tgt_set = set(tgt_idxs.tolist())
    src_set = set(src_idxs.tolist())

    # ── sim candidate edges ───────────────────────────────────────────────
    se = sim_edges.copy()
    se["src_pid"] = se["src_idx"].map(idx_to_pid)
    se["dst_pid"] = se["dst_idx"].map(idx_to_pid)
    se["label"]   = se.apply(
        lambda r: 1 if (r["src_pid"], r["dst_pid"]) in cite_set else 0, axis=1
    )

    # ── citation-only edges (in scope but not in sim candidates) ──────────
    sim_pair_set = set(zip(se["src_pid"], se["dst_pid"]))

    cite_only_rows = []
    for src_pid, dst_pid in cite_set:
        if (src_pid, dst_pid) in sim_pair_set:
            continue
        si = pid_to_idx.get(src_pid)
        di = pid_to_idx.get(dst_pid)
        if si is None or di is None:
            continue
        if si not in src_set or di not in tgt_set:
            continue
        if si == di:
            continue
        # Compute sim via dot product
        s = float(emb[si] @ emb[di])
        cite_only_rows.append({
            "src_idx": si, "dst_idx": di,
            "src_pid": src_pid, "dst_pid": dst_pid,
            "sim": s, "label": 1,
        })

    if cite_only_rows:
        ce = pd.DataFrame(cite_only_rows)
        edges = pd.concat([se, ce], ignore_index=True)
    else:
        edges = se.copy()

    # ── Compute features ──────────────────────────────────────────────────
    edges["year_src"]    = edges["src_pid"].map(pid_to_year)
    edges["year_dst"]    = edges["dst_pid"].map(pid_to_year)
    edges["year_gap"]    = (edges["year_src"] - edges["year_dst"]).astype(float)
    edges["year_gap_norm"] = edges["year_gap"] / window

    edges["topic_src"]   = edges["src_pid"].map(pid_to_topic)
    edges["topic_dst"]   = edges["dst_pid"].map(pid_to_topic)
    edges["same_topic"]  = (
        (edges["topic_src"] == edges["topic_dst"]) &
        (edges["topic_src"] != -1)
    ).astype(int)

    edges["country_src"] = edges["src_pid"].map(pid_to_country)
    edges["country_dst"] = edges["dst_pid"].map(pid_to_country)
    edges["country_pair"] = edges["country_src"] + "->" + edges["country_dst"]

    # Drop any edges outside CN/US (safety)
    edges = edges[edges["country_pair"].isin(
        {"CN->CN", "CN->US", "US->CN", "US->US"}
    )].copy()

    return edges


def year_gap_bin(g):
    if g <= 0:
        return "0"
    elif g <= 2:
        return "1-2"
    elif g <= 5:
        return "3-5"
    else:
        return ">5"


def matched_negative_sampling(edges, ratio=NEG_RATIO, seed=SEED):
    """Matched 1:1 negative sampling stratified by country_pair x year_gap_bin."""
    rng = np.random.RandomState(seed)

    pos = edges[edges["label"] == 1].copy()
    neg = edges[edges["label"] == 0].copy()

    if len(pos) == 0 or len(neg) == 0:
        return edges[edges["label"] == 1]  # return positives only

    pos["_ygb"] = pos["year_gap"].apply(year_gap_bin)
    neg["_ygb"] = neg["year_gap"].apply(year_gap_bin)

    sampled_neg = []
    for (cp, ygb), grp_pos in pos.groupby(["country_pair", "_ygb"]):
        n_want = int(len(grp_pos) * ratio)
        grp_neg = neg[(neg["country_pair"] == cp) & (neg["_ygb"] == ygb)]
        if len(grp_neg) == 0:
            continue
        n_take = min(n_want, len(grp_neg))
        sampled_neg.append(grp_neg.sample(n=n_take, random_state=rng))

    if sampled_neg:
        neg_sampled = pd.concat(sampled_neg, ignore_index=True)
    else:
        # Fallback: simple random sample
        n_want = int(len(pos) * ratio)
        neg_sampled = neg.sample(n=min(n_want, len(neg)), random_state=rng)

    result = pd.concat([pos, neg_sampled], ignore_index=True)
    result.drop(columns=["_ygb"], errors="ignore", inplace=True)
    return result


def check_minimum_data(edges):
    """Check if there is enough data to fit a model."""
    pos = edges[edges["label"] == 1]
    n_pos = len(pos)
    if n_pos < MIN_POS_TOTAL:
        return False, f"n_pos={n_pos} < {MIN_POS_TOTAL}"
    n_cn_us = len(pos[pos["country_pair"] == "CN->US"])
    n_us_cn = len(pos[pos["country_pair"] == "US->CN"])
    if n_cn_us < MIN_POS_PAIR:
        return False, f"n_pos_cn_us={n_cn_us} < {MIN_POS_PAIR}"
    if n_us_cn < MIN_POS_PAIR:
        return False, f"n_pos_us_cn={n_us_cn} < {MIN_POS_PAIR}"
    return True, "ok"


def fit_logit(edges):
    """Fit logistic regression and extract betas.
    Reference category: CN->CN (absorbed into intercept).
    Features:
      I(CN->US), I(US->CN), I(US->US),
      year_gap_norm, same_topic,
      sim*I(CN->CN), sim*I(CN->US), sim*I(US->CN), sim*I(US->US)

    Uses robust scaling safeguards for numerical stability, then converts
    coefficients back to the original (raw-cosine) scale for
    cross-year comparability.
    """
    e = edges.copy()

    # Dummies for country pair (CN->CN is reference)
    e["is_cn_us"] = (e["country_pair"] == "CN->US").astype(float)
    e["is_us_cn"] = (e["country_pair"] == "US->CN").astype(float)
    e["is_us_us"] = (e["country_pair"] == "US->US").astype(float)

    # Clip sim to [-1, 1] to guard against floating-point artefacts
    e["sim"] = e["sim"].clip(-1.0, 1.0)

    # Interaction: sim x country_pair
    e["sim_x_cncn"] = e["sim"] * (e["country_pair"] == "CN->CN").astype(float)
    e["sim_x_cnus"] = e["sim"] * e["is_cn_us"]
    e["sim_x_uscn"] = e["sim"] * e["is_us_cn"]
    e["sim_x_usus"] = e["sim"] * e["is_us_us"]

    feature_cols = [
        "is_cn_us", "is_us_cn", "is_us_us",
        "year_gap_norm", "same_topic",
        "sim_x_cncn", "sim_x_cnus", "sim_x_uscn", "sim_x_usus",
    ]
    X = e[feature_cols].values.astype(np.float64)
    y = e["label"].values.astype(int)

    # ── Drop rows with NaN / Inf ──────────────────────────────────────────
    finite_mask = np.isfinite(X).all(axis=1)
    X = X[finite_mask]
    y = y[finite_mask]
    if len(X) == 0 or len(np.unique(y)) < 2:
        raise ValueError("insufficient_finite_or_single_class")

    # ── Standardise for numerical stability ───────────────────────────────
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    # Catch constant or near-constant columns before division
    sd[~np.isfinite(sd) | (sd < 1e-8)] = 1.0
    X_scaled = (X - mu) / sd
    # Replace any residual non-finite values and clip tails for stable optimisation
    X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=8.0, neginf=-8.0)
    X_scaled = np.clip(X_scaled, -8.0, 8.0)

    # liblinear is more stable than lbfgs here under quasi-separation
    clf = LogisticRegression(
        max_iter=5000, C=1.0, class_weight="balanced",
        solver="liblinear", penalty="l2", random_state=SEED,
    )
    clf.fit(X_scaled, y)

    # ── Convert coefficients back to original (unscaled) space ────────────
    # If X' = (X - mu) / sd, model learns coef' on X'.
    # Original-scale: coef = coef' / sd,  intercept = intercept' - sum(coef' * mu / sd)
    coef_original = clf.coef_[0] / sd
    intercept_original = clf.intercept_[0] - np.dot(clf.coef_[0], mu / sd)

    coef_dict = dict(zip(feature_cols, coef_original))
    return {
        "beta_sim_cn_cn": coef_dict["sim_x_cncn"],
        "beta_sim_cn_us": coef_dict["sim_x_cnus"],
        "beta_sim_us_cn": coef_dict["sim_x_uscn"],
        "beta_sim_us_us": coef_dict["sim_x_usus"],
        "intercept":      intercept_original,
        "coef_is_cn_us":  coef_dict["is_cn_us"],
        "coef_is_us_cn":  coef_dict["is_us_cn"],
        "coef_is_us_us":  coef_dict["is_us_us"],
        "coef_year_gap":  coef_dict["year_gap_norm"],
        "coef_same_topic": coef_dict["same_topic"],
    }


def run_one_year(source_year, window, verbose=True):
    """Full pipeline for a single (source_year, window). Returns result dict."""
    src_idxs, tgt_idxs = get_source_and_target(source_year, window)
    n_src = len(src_idxs)

    if n_src == 0:
        if verbose:
            print(f"  y={source_year} w={window}: SKIP (no source papers)")
        return _make_nan_row(source_year, window, "no_source_papers")

    # Restricted similarity search
    sim_edges = restricted_sim_search(src_idxs, tgt_idxs)

    # Build union edges
    edges = build_union_edges(sim_edges, src_idxs, tgt_idxs, window)

    # Check minimum data
    ok, reason = check_minimum_data(edges)
    if not ok:
        if verbose:
            print(f"  y={source_year} w={window}: SKIP ({reason})")
        return _make_nan_row(source_year, window, reason, n_src=n_src,
                             n_pos=len(edges[edges["label"]==1]))

    # Matched negative sampling
    edges_sampled = matched_negative_sampling(edges)
    n_pos = int(edges_sampled["label"].sum())
    n_neg = int((edges_sampled["label"] == 0).sum())
    pos_sub = edges_sampled[edges_sampled["label"] == 1]
    n_pos_cn_us = int((pos_sub["country_pair"] == "CN->US").sum())
    n_pos_us_cn = int((pos_sub["country_pair"] == "US->CN").sum())

    # Fit logistic regression
    betas = fit_logit(edges_sampled)

    beta_gap = betas["beta_sim_cn_us"] - betas["beta_sim_us_cn"]

    if verbose:
        print(f"  y={source_year} w={window}: n_src={n_src:,}  "
              f"pos={n_pos}  neg={n_neg}  "
              f"β(CN→US)={betas['beta_sim_cn_us']:.3f}  "
              f"β(US→CN)={betas['beta_sim_us_cn']:.3f}  "
              f"gap={beta_gap:.3f}")

    return {
        "source_year": source_year,
        "window": window,
        "n_src_papers": n_src,
        "n_pos": n_pos,
        "n_neg": n_neg,
        "n_pos_cn_us": n_pos_cn_us,
        "n_pos_us_cn": n_pos_us_cn,
        "beta_sim_cn_cn": betas["beta_sim_cn_cn"],
        "beta_sim_cn_us": betas["beta_sim_cn_us"],
        "beta_sim_us_cn": betas["beta_sim_us_cn"],
        "beta_sim_us_us": betas["beta_sim_us_us"],
        "beta_gap_cnus_minus_uscn": beta_gap,
        "status": "ok",
    }


def _make_nan_row(source_year, window, reason, n_src=0, n_pos=0):
    return {
        "source_year": source_year,
        "window": window,
        "n_src_papers": n_src,
        "n_pos": n_pos,
        "n_neg": 0,
        "n_pos_cn_us": 0,
        "n_pos_us_cn": 0,
        "beta_sim_cn_cn": np.nan,
        "beta_sim_cn_us": np.nan,
        "beta_sim_us_cn": np.nan,
        "beta_sim_us_us": np.nan,
        "beta_gap_cnus_minus_uscn": np.nan,
        "status": reason,
    }


print("§2 Helper functions defined.")


§2 Helper functions defined.


## §3 Yearly fits for window = 3

For each `source_year` in [2005, 2024], fit the logistic model on the union edge set constructed from the 3-year target window `[y-2, y]`.

In [49]:
print("=" * 70)
print("§3  Window = 3")
print("=" * 70)

results_w3 = []
for y in range(SOURCE_YEAR_MIN, SOURCE_YEAR_MAX + 1):
    r = run_one_year(y, window=3)
    results_w3.append(r)

df_w3 = pd.DataFrame(results_w3)
print(f"\nDone. {(df_w3['status']=='ok').sum()} / {len(df_w3)} years fitted successfully.")

§3  Window = 3
  y=2005 w=3: SKIP (n_pos=5 < 30)
  y=2006 w=3: SKIP (n_pos=3 < 30)
  y=2007 w=3: SKIP (n_pos=8 < 30)
  y=2008 w=3: SKIP (n_pos=5 < 30)
  y=2009 w=3: SKIP (n_pos=17 < 30)
  y=2010 w=3: SKIP (n_pos=19 < 30)
  y=2011 w=3: SKIP (n_pos=27 < 30)
  y=2012 w=3: SKIP (n_pos_cn_us=2 < 5)
  y=2013 w=3: SKIP (n_pos_us_cn=1 < 5)
  y=2014 w=3: SKIP (n_pos_us_cn=0 < 5)
  y=2015 w=3: SKIP (n_pos_us_cn=1 < 5)
  y=2016 w=3: SKIP (n_pos_cn_us=2 < 5)
  y=2017 w=3: SKIP (n_pos_us_cn=3 < 5)
  y=2018 w=3: n_src=404  pos=175  neg=175  β(CN→US)=1.911  β(US→CN)=1.621  gap=0.290
  y=2019 w=3: n_src=473  pos=197  neg=197  β(CN→US)=2.637  β(US→CN)=1.846  gap=0.791
  y=2020 w=3: n_src=564  pos=328  neg=328  β(CN→US)=2.332  β(US→CN)=1.090  gap=1.243
  y=2021 w=3: n_src=615  pos=306  neg=306  β(CN→US)=0.649  β(US→CN)=0.975  gap=-0.326
  y=2022 w=3: n_src=801  pos=504  neg=504  β(CN→US)=-2.222  β(US→CN)=2.485  gap=-4.707
  y=2023 w=3: n_src=846  pos=631  neg=631  β(CN→US)=0.649  β(US→CN)=-0.035  gap=0.

## §4 Yearly fits for window = 5

Same pipeline but with 5-year target window `[y-4, y]`.

In [50]:
print("=" * 70)
print("§4  Window = 5")
print("=" * 70)

results_w5 = []
for y in range(SOURCE_YEAR_MIN, SOURCE_YEAR_MAX + 1):
    r = run_one_year(y, window=5)
    results_w5.append(r)

df_w5 = pd.DataFrame(results_w5)
print(f"\nDone. {(df_w5['status']=='ok').sum()} / {len(df_w5)} years fitted successfully.")

§4  Window = 5
  y=2005 w=5: SKIP (n_pos=7 < 30)
  y=2006 w=5: SKIP (n_pos=4 < 30)
  y=2007 w=5: SKIP (n_pos=10 < 30)
  y=2008 w=5: SKIP (n_pos=7 < 30)
  y=2009 w=5: SKIP (n_pos=19 < 30)
  y=2010 w=5: SKIP (n_pos=27 < 30)
  y=2011 w=5: SKIP (n_pos_cn_us=2 < 5)
  y=2012 w=5: SKIP (n_pos_cn_us=3 < 5)
  y=2013 w=5: SKIP (n_pos_us_cn=1 < 5)
  y=2014 w=5: SKIP (n_pos_us_cn=0 < 5)
  y=2015 w=5: SKIP (n_pos_us_cn=1 < 5)
  y=2016 w=5: n_src=281  pos=105  neg=105  β(CN→US)=1.890  β(US→CN)=1.303  gap=0.587
  y=2017 w=5: SKIP (n_pos_us_cn=3 < 5)
  y=2018 w=5: n_src=404  pos=264  neg=264  β(CN→US)=2.238  β(US→CN)=1.996  gap=0.242
  y=2019 w=5: n_src=473  pos=288  neg=288  β(CN→US)=3.158  β(US→CN)=2.224  gap=0.934
  y=2020 w=5: n_src=564  pos=483  neg=483  β(CN→US)=2.534  β(US→CN)=2.102  gap=0.432
  y=2021 w=5: n_src=615  pos=508  neg=508  β(CN→US)=0.130  β(US→CN)=-0.372  gap=0.502
  y=2022 w=5: n_src=801  pos=756  neg=756  β(CN→US)=-1.713  β(US→CN)=2.874  gap=-4.586
  y=2023 w=5: n_src=846  pos=98

## §4b Yearly fits for window = 10

Same pipeline but with 10-year target window `[y-9, y]`.

In [51]:
print("=" * 70)
print("§4b  Window = 10")
print("=" * 70)

results_w10 = []
for y in range(SOURCE_YEAR_MIN, SOURCE_YEAR_MAX + 1):
    r = run_one_year(y, window=10)
    results_w10.append(r)

df_w10 = pd.DataFrame(results_w10)
print(f"\nDone. {(df_w10['status']=='ok').sum()} / {len(df_w10)} years fitted successfully.")

§4b  Window = 10
  y=2005 w=10: SKIP (n_pos=8 < 30)
  y=2006 w=10: SKIP (n_pos=10 < 30)
  y=2007 w=10: SKIP (n_pos=14 < 30)
  y=2008 w=10: SKIP (n_pos=11 < 30)
  y=2009 w=10: SKIP (n_pos=26 < 30)
  y=2010 w=10: SKIP (n_pos=29 < 30)
  y=2011 w=10: SKIP (n_pos_cn_us=2 < 5)
  y=2012 w=10: SKIP (n_pos_us_cn=0 < 5)
  y=2013 w=10: SKIP (n_pos_us_cn=1 < 5)
  y=2014 w=10: SKIP (n_pos_us_cn=0 < 5)
  y=2015 w=10: SKIP (n_pos_us_cn=1 < 5)
  y=2016 w=10: n_src=281  pos=149  neg=149  β(CN→US)=1.559  β(US→CN)=0.958  gap=0.600
  y=2017 w=10: SKIP (n_pos_us_cn=3 < 5)
  y=2018 w=10: n_src=404  pos=357  neg=357  β(CN→US)=2.725  β(US→CN)=0.675  gap=2.051
  y=2019 w=10: n_src=473  pos=409  neg=409  β(CN→US)=2.385  β(US→CN)=1.373  gap=1.012
  y=2020 w=10: n_src=564  pos=675  neg=675  β(CN→US)=1.568  β(US→CN)=1.372  gap=0.195
  y=2021 w=10: n_src=615  pos=686  neg=686  β(CN→US)=0.186  β(US→CN)=-1.562  gap=1.748
  y=2022 w=10: n_src=801  pos=1054  neg=1054  β(CN→US)=-2.996  β(US→CN)=0.071  gap=-3.067
  y=202

## §5 Export tables and plots

In [52]:
# ── 5a  Export CSV tables ──────────────────────────────────────────────────
export_cols = [
    "source_year", "window", "n_src_papers",
    "n_pos", "n_neg", "n_pos_cn_us", "n_pos_us_cn",
    "beta_sim_cn_cn", "beta_sim_cn_us", "beta_sim_us_cn", "beta_sim_us_us",
    "beta_gap_cnus_minus_uscn", "status",
]

path_w3 = OUT_TABLE / "beta_by_year_window3.csv"
df_w3[export_cols].to_csv(path_w3, index=False)
print(f"Saved: {path_w3}")

path_w5 = OUT_TABLE / "beta_by_year_window5.csv"
df_w5[export_cols].to_csv(path_w5, index=False)
print(f"Saved: {path_w5}")

path_w10 = OUT_TABLE / "beta_by_year_window10.csv"
df_w10[export_cols].to_csv(path_w10, index=False)
print(f"Saved: {path_w10}")

# Long-format combined table
df_all = pd.concat([df_w3, df_w5, df_w10], ignore_index=True)
path_all = OUT_TABLE / "beta_by_year_all.csv"
df_all[export_cols].to_csv(path_all, index=False)
print(f"Saved: {path_all}")

df_all[df_all["status"] == "ok"][export_cols].head(10)

Saved: output/compare_networks/tables/beta_by_year_window3.csv
Saved: output/compare_networks/tables/beta_by_year_window5.csv
Saved: output/compare_networks/tables/beta_by_year_window10.csv
Saved: output/compare_networks/tables/beta_by_year_all.csv


,source_year,window,n_src_papers,n_pos,n_neg,n_pos_cn_us,n_pos_us_cn,beta_sim_cn_cn,beta_sim_cn_us,beta_sim_us_cn,beta_sim_us_us,beta_gap_cnus_minus_uscn,status
13,2018,3,404,175,175,13,8,3.722571,1.911364,1.620972,4.332633,0.290392,ok
14,2019,3,473,197,197,18,5,3.723738,2.636908,1.846062,4.065295,0.790846,ok
15,2020,3,564,328,328,14,7,4.239964,2.332433,1.089708,4.291036,1.242725,ok
16,2021,3,615,306,306,20,8,2.555691,0.649498,0.975139,3.269650,-0.325641,ok
17,2022,3,801,504,504,29,10,-0.203614,-2.221681,2.485340,2.627344,-4.707022,ok
18,2023,3,846,631,631,38,10,2.270374,0.648907,-0.034815,4.295906,0.683722,ok
19,2024,3,1031,827,827,39,19,-0.580246,-0.085393,-3.301615,4.874915,3.216222,ok
31,2016,5,281,105,105,10,5,2.255724,1.890462,1.303370,2.175167,0.587091,ok
33,2018,5,404,264,264,20,9,3.982599,2.237595,1.995734,4.195458,0.241861,ok
34,2019,5,473,288,288,42,9,4.135612,3.158046,2.224345,4.401833,0.933701,ok


In [53]:
# ── 5b  Plot 1: beta time series – window = 3 ────────────────────────────
ok3 = df_w3[df_w3["status"] == "ok"].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ok3["source_year"], ok3["beta_sim_cn_us"], "o-", label=r"$\beta$(sim $\times$ CN$\to$US)", color="tab:red")
ax.plot(ok3["source_year"], ok3["beta_sim_us_cn"], "s-", label=r"$\beta$(sim $\times$ US$\to$CN)", color="tab:blue")
ax.set_xlabel("Source year")
ax.set_ylabel(r"$\beta$ (logistic coefficient)")
ax.set_title("Pair-specific similarity slopes — 3-year window")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / "beta_time_series_window3.png", dpi=150)
print(f"Saved: {OUT_FIG / 'beta_time_series_window3.png'}")
plt.show()

Saved: output/compare_networks/figs/beta_time_series_window3.png


/var/folders/0m/2sbyxxc10czg8dp3clzyzxym0000gn/T/ipykernel_44808/382817302.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [54]:
# ── 5c  Plot 2: beta time series – window = 5 ────────────────────────────
ok5 = df_w5[df_w5["status"] == "ok"].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ok5["source_year"], ok5["beta_sim_cn_us"], "o-", label=r"$\beta$(sim $\times$ CN$\to$US)", color="tab:red")
ax.plot(ok5["source_year"], ok5["beta_sim_us_cn"], "s-", label=r"$\beta$(sim $\times$ US$\to$CN)", color="tab:blue")
ax.set_xlabel("Source year")
ax.set_ylabel(r"$\beta$ (logistic coefficient)")
ax.set_title("Pair-specific similarity slopes — 5-year window")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / "beta_time_series_window5.png", dpi=150)
print(f"Saved: {OUT_FIG / 'beta_time_series_window5.png'}")
plt.show()

Saved: output/compare_networks/figs/beta_time_series_window5.png


/var/folders/0m/2sbyxxc10czg8dp3clzyzxym0000gn/T/ipykernel_44808/2824138476.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [55]:
# ── 5d  Plot 3: beta time series – window = 10 ───────────────────────────
ok10 = df_w10[df_w10["status"] == "ok"].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ok10["source_year"], ok10["beta_sim_cn_us"], "o-", label=r"$\beta$(sim $\times$ CN$\to$US)", color="tab:red")
ax.plot(ok10["source_year"], ok10["beta_sim_us_cn"], "s-", label=r"$\beta$(sim $\times$ US$\to$CN)", color="tab:blue")
ax.set_xlabel("Source year")
ax.set_ylabel(r"$\beta$ (logistic coefficient)")
ax.set_title("Pair-specific similarity slopes — 10-year window")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / "beta_time_series_window10.png", dpi=150)
print(f"Saved: {OUT_FIG / 'beta_time_series_window10.png'}")
plt.show()

Saved: output/compare_networks/figs/beta_time_series_window10.png


/var/folders/0m/2sbyxxc10czg8dp3clzyzxym0000gn/T/ipykernel_44808/3823254880.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [56]:
# ── 5e  Plot 4: beta gap (CN→US minus US→CN) for all windows ─────────────
fig, ax = plt.subplots(figsize=(10, 5))

if len(ok3) > 0:
    ax.plot(ok3["source_year"], ok3["beta_gap_cnus_minus_uscn"],
            "o-", label="window = 3", color="tab:orange")
if len(ok5) > 0:
    ax.plot(ok5["source_year"], ok5["beta_gap_cnus_minus_uscn"],
            "s-", label="window = 5", color="tab:green")
if len(ok10) > 0:
    ax.plot(ok10["source_year"], ok10["beta_gap_cnus_minus_uscn"],
            "^-", label="window = 10", color="tab:purple")

ax.axhline(0, color="grey", ls="--", lw=0.8)
ax.set_xlabel("Source year")
ax.set_ylabel(r"$\beta$(sim$\times$CN$\to$US) $-$ $\beta$(sim$\times$US$\to$CN)")
ax.set_title(r"$\beta$ gap: CN$\to$US minus US$\to$CN")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / "beta_gap_time_series.png", dpi=150)
print(f"Saved: {OUT_FIG / 'beta_gap_time_series.png'}")
plt.show()

Saved: output/compare_networks/figs/beta_gap_time_series.png


/var/folders/0m/2sbyxxc10czg8dp3clzyzxym0000gn/T/ipykernel_44808/3202754358.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## §6 Short interpretation

**How to read the results:**

- **beta(sim x CN->US)**: The marginal effect of embedding similarity on the log-odds of a Chinese paper citing an American paper. A higher value means Chinese researchers are more likely to cite similar American work (controlling for other factors).
- **beta(sim x US->CN)**: The analogous coefficient for American papers citing Chinese work.
- **beta_gap > 0**: Chinese papers respond more strongly to similarity when citing US work than vice versa. This suggests CN researchers are more actively incorporating relevant US knowledge than US researchers are incorporating CN knowledge.
- **beta_gap < 0**: The reverse — US researchers are more responsive to similarity when citing CN work.

**Temporal trends** in beta_gap reveal whether the asymmetry in cross-border knowledge absorption is widening, narrowing, or stable over time.

**Caveats:**
- Years with insufficient cross-border citations are skipped (NaN).
- Raw cosine similarity is used (no z-score) to ensure cross-year comparability.
- No uncertainty bands — interpret small year-to-year fluctuations with caution.
- The model controls for year_gap_norm and same_topic, but not for journal prestige, author reputation, or other confounders.